# Review all six canonical variables in one run

This batch workflow is separate from `1.variable_decide.ipynb`, which still
supports reviewing one variable. Run sections 1–3 **once** to check all six
variables and save all plots and readouts. Inspect the outputs, edit the manual
choices in section 4, and run section 5 to save final rules. Editing manual choices
requires rerunning only sections 4–5, with no database queries or plotting.


## 1. Setup and settings


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sqlalchemy as sa
from IPython.display import display

HERE = Path.cwd()
if not (HERE / 'func' / 'func_variable_mapping.py').exists():
    HERE = HERE / 'Quanlity_Control' / 'variable_mapping'
if not (HERE / 'func' / 'func_variable_mapping.py').exists():
    raise FileNotFoundError('Run from the repository root or variable_mapping folder.')
sys.path.insert(0, str(HERE / 'func'))
from func_variable_mapping import CANONICAL_VARIABLES
from func_variable_review import (
    run_variable_reviews, save_review_rules, format_candidate_review,
)

DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    '&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass',
)
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
NETWORK_ID = 2  # Change only this value to work on another network.
NETWORK_DIR = HERE / 'networks' / f'network_{NETWORK_ID}'
NETWORK_DIR.mkdir(parents=True, exist_ok=True)
VARIABLES_TO_REVIEW = list(CANONICAL_VARIABLES)  # All six in one run.
PLOT_STATION_LIMIT = 10
SOURCE_STATION_LIMIT = 20
RANDOM_SEED = 42  # None for a fresh sample; integer for repeatable samples.
AUTO_KEEP_FROM = '2026-01-01'
REGISTRY_PATH = NETWORK_DIR / 'candidate_mappings.csv'
FINAL_RULE_PATH = NETWORK_DIR / 'final_variable_rules.csv'

## 2. Run all variables once

For each canonical variable, the runner finds stations containing **every candidate
ID**, then samples up to 20 of those stations for recent-data checks and up to 10
for plots. The two samples are drawn separately; a fixed seed can make them overlap.
Eligibility requires observations at any date, including null values; timestamps
need not match between candidates. The recent-data check requires non-null raw
observations **on or after January 1, 2026**, before plausible-value filtering.

Automatic keeps, station counts, and latest observation dates describe the checked
sample, not the entire network. Eligible-station discovery still checks the network,
but is done only once per variable; recent-data queries use only the sampled stations.
A variable with missing candidates, no eligible stations, or no plausible plot data is
reported in the run summary. Unexpected database errors stop the run and are shown.

Each successful run replaces `networks/network_<id>/latest_review/` for the selected
network. Results for other networks are preserved. If a run fails, the previous
successful `latest_review/` remains available:

- `time_series.pdf`: all variables, one station per page.
- `cdf.pdf`: all variables, one station per page. Precipitation and snowfall use
  positive values on a logarithmic axis; other variables use all plausible values.
- `candidate_review.csv`: candidate counts, latest sample dates, and automatic decisions.
- `station_conditions.csv`: per-station recent-data conditions, organized by variable ID.
- `qc_summary.csv`: plausible-value filtering readouts.
- `run_summary.csv`: status and station selection for each canonical variable.
- `review_results.json`: saved results needed to apply manual choices later.

PDF pages include the canonical variable, station name/ID, and candidate ID legends.
A missing-data page explains when a variable has no plots. No final rules are saved yet.


In [ ]:
candidate_mappings = pd.read_csv(REGISTRY_PATH)
batch_results = run_variable_reviews(
    engine, candidate_mappings, NETWORK_DIR,
    network_id=NETWORK_ID,
    variables=VARIABLES_TO_REVIEW,
    plot_station_limit=PLOT_STATION_LIMIT,
    source_station_limit=SOURCE_STATION_LIMIT,
    random_seed=RANDOM_SEED,
    auto_keep_from=AUTO_KEEP_FROM,
)
RUN_DIR = Path(batch_results['run_dir'])

## 3. Inspect saved readouts and figures

This cell only reads saved files; rerunning it does not query the database.
Counts and latest dates apply to the source-check sample. `source_checked_at_utc`
is the time that check finished. Empty latest dates mean no qualifying observation
was found in the sample, which does not prove absence elsewhere in the network.


In [ ]:
from IPython.display import FileLink

display(pd.read_csv(RUN_DIR / 'run_summary.csv'))
source_review = format_candidate_review(
    pd.read_csv(RUN_DIR / 'candidate_review.csv'))
display(source_review.sort_values(['canonical_variable', 'vars_id']).set_index(
    ['canonical_variable', 'vars_id']
))
for filename in ['time_series.pdf', 'cdf.pdf', 'candidate_review.csv',
                 'station_conditions.csv', 'qc_summary.csv', 'run_summary.csv']:
    display(FileLink(str((RUN_DIR / filename).relative_to(Path.cwd()))
                     if (RUN_DIR / filename).is_relative_to(Path.cwd())
                     else str(RUN_DIR / filename)))

## 4. Edit manual source IDs after reviewing the outputs

Add candidate IDs you want to keep **in addition to** automatic keeps. Each key is a
canonical variable; each value is a list of IDs. Empty lists keep only automatic IDs.
For example, change `'daily_min_temperature': []` to
`'daily_min_temperature': [435]` if that ID is a candidate you want to retain.
All choices are validated before saving. This cell and the next may be rerun freely.


In [ ]:
MANUAL_SOURCE_IDS = {
    'air_temperature': [434],
    'daily_min_temperature': [435],
    'daily_max_temperature': [433],
    'precipitation_amount': [442],
    'snowfall_amount': [451],
    'snow_depth': [450],
}

## 5. Save final rules without rerunning the review

The save function combines automatic and manual IDs and updates one row per network
and canonical variable in the existing rules CSV. Other rules are preserved. Variables
with no selected IDs leave their existing rules unchanged. The same compact columns as
the single-variable workflow are used: sampled station IDs, the single check timestamp,
recent source IDs/latest dates, and merge policy. Detailed conditions and counts remain
in separate readouts. `final_rules.csv` in this run's folder records the saved selections.

After a kernel restart, rerun sections 1–3 before editing section 4 and saving. Only
the latest successful review is retained for each network, so this notebook does not
rely on previous-run folders. Saving itself does not connect to the database.


In [ ]:
final_rules = save_review_rules(
    batch_results, MANUAL_SOURCE_IDS, FINAL_RULE_PATH)
display(final_rules)
if FINAL_RULE_PATH.exists():
    print(f'Rules file: {FINAL_RULE_PATH}')
run_rules_path = RUN_DIR / 'final_rules.csv'
if run_rules_path.exists():
    print(f'Run-specific selections: {run_rules_path}')
else:
    print('No rules updated: no source IDs were selected for this run.')